In [ ]:
from scipy import signal
import numpy as np
import pandas as pd

# butterworth_filter

fs = 1000
bf = signal.butter(
    4,
    [20, 400],
    btype='bandpass',
    fs=fs,  
    output='sos'
)

#intialization of Global variables 

cols = ["Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6", "Ch7", "Ch8"]
fs = 1000
window_size = 250     # 250 samples = 250 ms
step_size = 250       # No overlap  

# feature extraction functions 

def hjorth_features(signal):
    x = np.asarray(signal, dtype=float)
    dx = np.diff(x)
    ddx = np.diff(dx)
    activity = np.var(x)
    mobility = np.sqrt(np.var(dx) / activity) if activity > 0 else 0
    mobility_dx = (np.sqrt(np.var(ddx) / np.var(dx))
        if np.var(dx) > 0 else 0 )
    complexity = mobility_dx / mobility if mobility > 0 else 0
    return activity, mobility, complexity

def calculate_mav(signal):
    return np.mean(np.abs(signal))

def calculate_rms(signal):
    return np.sqrt(np.mean(signal ** 2))

def calculate_waveform_length(signal):
    return np.sum(np.abs(np.diff(signal)))

def calculate_ssc(signal):
    diff1 = np.diff(signal)
    return np.sum(diff1[:-1] * diff1[1:] < 0)  

def calculate_variance(signal):
    return np.var(signal)

def calculate_iemg(signal):
    return np.sum(np.abs(signal))    

def Zero_Crossing(signal, threshold=0.01):
    signal = np.asarray(signal)

    crossings = ( (signal[:-1] * signal[1:] < 0) &
        (np.abs(signal[1:] - signal[:-1]) >= threshold) )

    return int(np.sum(crossings))    
    

# begin in the loop for all subjects , with all levels , with all repetitions, for all channels , filtering first , normalizing then windowing 
# and then feature extraction using the defined functions in the begininng

for c in [1,2,3,4,5,6,7] :

    for s in [1,2,3,4,5,6,7,8,9,10]:
            
        for f in [20, 30, 40, 50, 60, 70, 80]:
            
            for n in range(1, 5):
                file_name = f"Downloads/ContractionIntensity-main/ContractionIntensity-main/S{s}/S{s}_{f}P_C{c}_R{n}.csv"
                
                df = pd.read_csv(file_name, names=cols)
    
                filtered_df = pd.DataFrame()    
        
                for i in range(1, 9):
                    original_channel = f"Ch{i}"
                    filtered_channel = f"Ch{i}"
                    signal_data = pd.to_numeric(df[original_channel], errors="coerce").to_numpy()
                    filtered_df[filtered_channel] = signal.sosfilt(bf, signal_data)
                    
                output_file = f"all_files/Filtered/S{s}_{f}P_C{c}_R{n}_filtered.csv"
                
                filtered_df.to_csv(output_file, index=False)
        
                file_path = f"all_files/Filtered/S{s}_{f}P_C{c}_R{n}_filtered.csv"
                
                df = pd.read_csv(file_path)
                
                #normalized_data = pd.DataFrame() 
                
                #for j in range(1, 9):
                    #normalized_data[f"Ch{j}"] = df[f"Ch{j}"] / mvc_data.loc[j, f"S{s}"]
                # use dierctly the normalized data 
                #df = normalized_data    
                features = []     
                window_number = 1
                channels = [f"Ch{i}" for i in range(1,9)]
                for start in range(0, len(df) - window_size + 1, step_size):
                    end = start + window_size
                    row = {"Window": window_number,
                        "Start_Sample": start,
                        "End_Sample": end - 1,
                        "MVC_Percent": f }
                    for channel in channels:
                        window = df[channel].iloc[start:end].to_numpy()
                        # Extract features
                        row[f"{channel}_MAV"] = calculate_mav(window)
                        row[f"{channel}_RMS"] = calculate_rms(window)
                        row[f"{channel}_WL"] = calculate_waveform_length(window)
                        row[f"{channel}_ssc"] = calculate_ssc(window)
                        row[f"{channel}_iemg"] = calculate_iemg(window)
                        row[f"{channel}_variance"] = calculate_variance(window)
                        (row[f"{channel}_Hjorth_variance"] , 
                        row[f"{channel}_Hjorth_mobility"], 
                        row[f"{channel}_Hjorth_complexity"],)=  hjorth_features(window)
                        row[f"{channel}_ZC"] = Zero_Crossing(window)
                        
                    features.append(row)
                
                    window_number += 1
                
                features_df = pd.DataFrame(features)
                
                output_file = f"all_files/Features/S{s}_{f}P_C{c}_R{n}_8Ch_features.csv"
                
                features_df.to_csv(output_file, index=False)
                
                print("\nSaved to:", output_file)

In [12]:
import pandas as pd
from pathlib import Path
folder = Path("all_files/Features")
for c in [1,2,3,4,5,6,7]:
    for s in [1,2,3,4,5,6,7,8,9,10]:     
        dataframes = []
        files = [
            folder / f"S{s}_50P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_80P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_70P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_40P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_60P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_20P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_80P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_60P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_50P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_40P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_70P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_20P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_50P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_30P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_80P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_40P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_60P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_30P_C{c}_R2_8Ch_features.csv",
            folder / f"S{s}_70P_C{c}_R4_8Ch_features.csv",
            folder / f"S{s}_30P_C{c}_R1_8Ch_features.csv",
            folder / f"S{s}_20P_C{c}_R1_8Ch_features.csv",]
        
        for file in files:
            df = pd.read_csv(file)
            dataframes.append(df)
            # Combine all DataFrames
        data = pd.concat(dataframes, ignore_index=True)
        data.to_csv(f"all_files/Train/Dataset_S{s}_C{c}_features.csv",index=False)
        

In [13]:
for c in [1,2,3,4,5,6,7]:
    for s in [1,2,3,4,5,6,7,8,9,10]:
        dataframes = []
        files = [
            folder / f"S{s}_40P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_50P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_60P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_70P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_80P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_20P_C{c}_R3_8Ch_features.csv",
            folder / f"S{s}_30P_C{c}_R3_8Ch_features.csv",]
        
        for file in files:
            df = pd.read_csv(file)
            dataframes.append(df)
        data_test = pd.concat(dataframes, ignore_index=True)
        output_file = f"all_files/Test/Test_Dataset_S{s}_C{c}_features.csv"
        data_test.to_csv(output_file,index=False)
        print ("/n saved to :",output_file)

/n saved to : all_files/Test/Test_Dataset_S1_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S2_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S3_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S4_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S5_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S6_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S7_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S8_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S9_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S10_C1_features.csv
/n saved to : all_files/Test/Test_Dataset_S1_C2_features.csv
/n saved to : all_files/Test/Test_Dataset_S2_C2_features.csv
/n saved to : all_files/Test/Test_Dataset_S3_C2_features.csv
/n saved to : all_files/Test/Test_Dataset_S4_C2_features.csv
/n saved to : all_files/Test/Test_Dataset_S5_C2_features.csv
/n saved to : all_files/Test/Test_Dataset_S6_C2_features.csv
/n saved to : all_files

In [7]:
import pandas as pd
c=4
s=1
df_train=pd.read_csv(f"Features/Train&Test/Dataset_S{s}_C{c}_features_Ohne_normalization.csv") 
df_test=pd.read_csv(f"Features/Train&Test/Test_Dataset_S{s}_C{c}_features_Ohne_normalization.csv")
print("Original training samples:", len(df_train))
data_train = df_train.sample(frac=1,random_state=42).reset_index(drop=True)
data_test=df_test
print("Test samples:", len(df_test))
feature_types = ("_peak_to_peak","_MAV","_RMS","_iemg","_variance","_Diff_Max_Min","_ssc","_WL")
feature_columns = [
    col for col in data_train.columns
    if col.endswith(feature_types)
]
X_train = data_train[feature_columns]
X_test = data_test[feature_columns]
y_train = data_train["MVC_Percent"]
y_test = data_test["MVC_Percent"]

print("\nNumber of input features:", len(feature_columns))
print("Feature names:")
print(feature_columns)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Original training samples: 250
Test samples: 84

Number of input features: 64
Feature names:
['Ch1_MAV', 'Ch1_RMS', 'Ch1_WL', 'Ch1_ssc', 'Ch1_iemg', 'Ch1_variance', 'Ch1_Diff_Max_Min', 'Ch1_peak_to_peak', 'Ch2_MAV', 'Ch2_RMS', 'Ch2_WL', 'Ch2_ssc', 'Ch2_iemg', 'Ch2_variance', 'Ch2_Diff_Max_Min', 'Ch2_peak_to_peak', 'Ch3_MAV', 'Ch3_RMS', 'Ch3_WL', 'Ch3_ssc', 'Ch3_iemg', 'Ch3_variance', 'Ch3_Diff_Max_Min', 'Ch3_peak_to_peak', 'Ch4_MAV', 'Ch4_RMS', 'Ch4_WL', 'Ch4_ssc', 'Ch4_iemg', 'Ch4_variance', 'Ch4_Diff_Max_Min', 'Ch4_peak_to_peak', 'Ch5_MAV', 'Ch5_RMS', 'Ch5_WL', 'Ch5_ssc', 'Ch5_iemg', 'Ch5_variance', 'Ch5_Diff_Max_Min', 'Ch5_peak_to_peak', 'Ch6_MAV', 'Ch6_RMS', 'Ch6_WL', 'Ch6_ssc', 'Ch6_iemg', 'Ch6_variance', 'Ch6_Diff_Max_Min', 'Ch6_peak_to_peak', 'Ch7_MAV', 'Ch7_RMS', 'Ch7_WL', 'Ch7_ssc', 'Ch7_iemg', 'Ch7_variance', 'Ch7_Diff_Max_Min', 'Ch7_peak_to_peak', 'Ch8_MAV', 'Ch8_RMS', 'Ch8_WL', 'Ch8_ssc', 'Ch8_iemg', 'Ch8_variance', 'Ch8_Diff_Max_Min', 'Ch8_peak_to_peak']

Training samples:

In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

data = {
    "Model": ["Gradient Boosting","Random forest","Extra tree", "KNN","Linear regresion" ,"ridge regression"],
    "MAE": [],
    "RMSE":[],
    "R2": []
}

# 2 Gradient_boosting
from sklearn.ensemble import GradientBoostingRegressor
model_gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42)

model_gb.fit(X_train, y_train)

y_pred_gb = model_gb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_gb)

rmse = mean_squared_error(y_test, y_pred_gb) ** 0.5

r2 = r2_score(y_test, y_pred_gb)

data["MAE"].append(mae)
data["RMSE"].append(rmse)
data["R2"].append(r2)

# Create the Random Forest model
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_rf)

rmse = mean_squared_error(y_test, y_pred_rf) ** 0.5

r2 = r2_score(y_test, y_pred_rf)

data["MAE"].append(mae)
data["RMSE"].append(rmse)
data["R2"].append(r2)

#3 Extra trees
from sklearn.ensemble import ExtraTreesRegressor

model_et = ExtraTreesRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_et.fit(X_train, y_train)

y_pred_et = model_et.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_et)

rmse = mean_squared_error(y_test, y_pred_et) ** 0.5

r2 = r2_score(y_test, y_pred_et)

data["MAE"].append(mae)
data["RMSE"].append(rmse)
data["R2"].append(r2)

#knn
from sklearn.neighbors import KNeighborsRegressor

model_knn = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=5)
)

model_knn.fit(X_train, y_train)

y_pred_knn = model_knn.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_knn)

rmse = mean_squared_error(y_test, y_pred_knn) ** 0.5

r2 = r2_score(y_test, y_pred_knn)

data["MAE"].append(mae)
data["RMSE"].append(rmse)
data["R2"].append(r2)

#linear regression

from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()

model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_lr)

rmse = mean_squared_error(y_test, y_pred_lr) ** 0.5

r2 = r2_score(y_test, y_pred_lr)

data["MAE"].append(mae)
data["RMSE"].append(rmse)
data["R2"].append(r2)
# ridge regression
from sklearn.linear_model import Ridge

model_ridge = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

model_ridge.fit(X_train, y_train)

y_pred_ridge = model_ridge.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_ridge)

rmse = mean_squared_error(y_test, y_pred_ridge) ** 0.5

r2 = r2_score(y_test, y_pred_ridge)

data["MAE"].append(mae)
data["RMSE"].append(rmse)
data["R2"].append(r2)
df = pd.DataFrame(data)
print(df)
df.to_excel(f"Features/Excel_files/results_S{s}_C6.xlsx", index=False)

               Model       MAE      RMSE        R2
0  Gradient Boosting  3.844462  4.511339  0.949120
1      Random forest  4.167857  4.881196  0.940435
2         Extra tree  3.876190  4.770588  0.943104
3                KNN  3.380952  4.546061  0.948333
4   Linear regresion  2.841148  3.586084  0.967850
5   ridge regression  2.590968  3.297285  0.972820
